<!--nav--> [🗺 Learning path](README.md) · **37/39** · ◀ [RAG & Agent Serving Patterns](./RAG_Agent_Serving_Patterns.ipynb) · [Anatomy of a Decode Step](./Anatomy_Of_A_Decode_Step.ipynb) ▶

# Production Hardening: Cancellation, Backpressure, Drain & Chaos

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sugeerth/gpu-training-notebooks/blob/main/Production_Hardening_Reliability.ipynb)

The track so far has made your serving stack **fast**. This notebook makes it **survivable** — the
difference between a demo and a system that stays up on its worst day.

LLM serving has reliability problems that ordinary web services don't: requests run for *minutes*,
hold scarce GPU memory the whole time, stream partial results that can't be retried idempotently,
and degrade in ways that look fine in a health check.

| Part | What you'll learn |
|---|---|
| **1** | **Cancellation** — the KV leak that silently eats your capacity |
| **2** | **Backpressure**: why an unbounded queue is worse than a 429 |
| **3** | **Per-tenant fairness** — stopping one customer from consuming everyone's GPU |
| **4** | **Graceful drain & rollout** — deploying without dropping streams |
| **5** | The **failure taxonomy**: what actually breaks, and what each looks like in the logs |
| **6** | **A chaos drill, simulated**: lose a replica at peak and watch the SLO |
| **7** | The runbook |

**Runs on:** any CPU.

In [ ]:
import math, json, random, uuid, statistics
from collections import defaultdict
from IPython.display import HTML, display

D3_URL = "https://cdn.jsdelivr.net/npm/d3@7/dist/d3.min.js"
def show_d3(js, data=None, height=420):
    div = f"viz_{uuid.uuid4().hex[:10]}"
    html = f'''
<div id="{div}" style="width:100%;max-width:920px;font-family:system-ui,sans-serif"></div>
<script>
(function() {{
  function run() {{
    const d3 = window.d3, root = d3.select("#{div}"), data = {json.dumps(data)};
    const W = (document.getElementById("{div}").clientWidth || 880), H = {height};
    try {{ {js} }} catch (e) {{ root.append("pre").style("color","crimson").text("viz error: " + e); }}
  }}
  if (window.d3) run();
  else {{ const s = document.createElement("script"); s.src = "{D3_URL}"; s.onload = run;
          s.onerror = () => document.getElementById("{div}").textContent = "Could not load D3.";
          document.head.appendChild(s); }}
}})();
</script>'''
    display(HTML(html))
print("ready")

## Part 1 · Cancellation: the leak nobody notices

A user closes the browser tab mid-stream. What happens?

```
   client disconnects
        │
        ├─ HANDLED:  server notices → aborts the request → KV blocks freed immediately ✅
        │
        └─ UNHANDLED: generation runs to max_tokens, holding KV the entire time ❌
                      nobody is reading the output. You are computing for no one.
```

The second case is remarkably common because **nothing looks broken**: no errors, no alerts, and
throughput metrics even look *healthy* — you're generating plenty of tokens. What you lose is
**capacity**, which shows up as mysterious queueing (nb 26) that doesn't match your traffic.

Modern engines abort on disconnect when the HTTP layer propagates it, but the failure modes cluster
at the edges: proxies that buffer responses, load balancers that hold connections open, and
client libraries that don't close streams. Let's price the leak:

In [ ]:
def leak_cost(abandon_rate, max_tokens=512, typical_read=180, slots=64,
              decode_step_ms=25):
    # Abandoned requests keep a slot until max_tokens instead of until the user stopped reading.
    wasted_tokens = (max_tokens - typical_read) * abandon_rate
    useful_tokens = typical_read
    waste_frac = wasted_tokens / (useful_tokens + wasted_tokens)
    effective_slots = slots * (1 - waste_frac)
    return {"abandon_rate": abandon_rate, "waste_frac": waste_frac,
            "effective_slots": effective_slots,
            "capacity_lost": slots - effective_slots}

print("64 scheduler slots, max_tokens=512, users typically read ~180 tokens before leaving:\n")
print(f"{'abandon rate':>14}{'wasted work':>14}{'effective slots':>18}{'capacity lost':>15}")
print("-" * 62)
for rate in (0.0, 0.02, 0.05, 0.10, 0.20, 0.35):
    r = leak_cost(rate)
    print(f"{rate:>13.0%}{r['waste_frac']:>14.1%}{r['effective_slots']:>18.1f}"
          f"{r['capacity_lost']:>14.1f}")

print("\nA 10% abandon rate with unhandled cancellation costs ~16% of your fleet.")
print("You would never buy 16% fewer GPUs on purpose, but this is the same thing.")
print("\nHow to verify it's working (a 60-second test you should run today):")
print("  1. start a long streaming request, then kill the client mid-stream")
print("  2. watch `vllm:num_requests_running` in /metrics (nb 26)")
print("  3. it should drop within a second or two. If it stays elevated until the")
print("     generation would have finished, your cancellation path is broken.")
print("\nUsual culprits: a buffering reverse proxy (nginx proxy_buffering on),")
print("an API gateway that reads the full response before forwarding, or a client")
print("that abandons the socket without closing it.")

## Part 2 · Backpressure: bounded queues and honest 429s

Notebook 27 showed what happens past the knee: latency explodes. The instinct is to queue everything
and let it drain. That instinct is wrong, for a reason worth stating precisely:

> **A request queued longer than the client's timeout is worse than a rejected request.** You spend
> GPU on work nobody will receive, *and* the user still gets an error — just later.

The right shape is **admission control**:

```
   arrival ──► is queue depth < limit?  ──no──► 429 + Retry-After   (fast, honest, cheap)
                        │
                       yes
                        ▼
                     enqueue ──► serve
```

The queue limit should derive from your SLO, not from a round number:

$$\text{max queue depth} \approx \text{service rate} \times \text{acceptable wait}$$

In [ ]:
def admission(arrival_rate, service_rate, client_timeout=30.0, sim_seconds=300,
              queue_limit=None, seed=0):
    rng = random.Random(seed)
    queue, served, rejected, timed_out, wasted = [], 0, 0, 0, 0.0
    t = 0.0
    dt = 0.05
    while t < sim_seconds:
        # Poisson arrivals
        n_arrive = 0
        p = math.exp(-arrival_rate * dt); acc = rng.random()
        while acc > p:
            n_arrive += 1; acc *= rng.random()
        for _ in range(n_arrive):
            if queue_limit is not None and len(queue) >= queue_limit:
                rejected += 1                                  # fast 429, no GPU spent
            else:
                queue.append(t)
        # service
        capacity = service_rate * dt
        while queue and capacity >= 1:
            born = queue.pop(0)
            wait = t - born
            if wait > client_timeout:
                timed_out += 1                                 # served, but nobody was listening
                wasted += 1
            else:
                served += 1
            capacity -= 1
        t += dt
    total = served + rejected + timed_out
    return {"served": served, "rejected": rejected, "timed_out": timed_out,
            "wasted_gpu_frac": wasted / max(served + timed_out, 1),
            "good_frac": served / max(total, 1)}

SERVICE = 6.0     # requests/s this replica can retire (nb 27's goodput)
print(f"Replica capacity {SERVICE} req/s, client timeout 30s, 300s of overload at 9 req/s:\n")
print(f"{'policy':<28}{'served OK':>11}{'429s':>8}{'timed out':>11}{'wasted GPU':>12}")
print("-" * 72)
for label, limit in (("unbounded queue", None), ("queue limit 500", 500),
                     ("queue limit 180 (=30s)", 180), ("queue limit 60 (=10s)", 60)):
    r = admission(9.0, SERVICE, queue_limit=limit)
    print(f"{label:<28}{r['served']:>11}{r['rejected']:>8}{r['timed_out']:>11}"
          f"{r['wasted_gpu_frac']:>11.1%}")

print("\nThe unbounded queue serves the FEWEST users successfully while burning GPU on")
print("requests that already timed out. A bounded queue converts invisible waste into")
print("visible, actionable 429s - which clients can retry with backoff, or shed.")
print("\nRule of thumb: queue_limit ≈ service_rate × acceptable_wait.")
print(f"  here: {SERVICE} req/s × 30s = {int(SERVICE*30)} requests")
print("\nAlways send `Retry-After` with a 429. A client that retries immediately in a tight")
print("loop turns your backpressure into a self-inflicted DDoS.")

## Part 3 · Per-tenant fairness

Notebook 30 put many tenants on one GPU. Without limits, **one tenant's batch job consumes the whole
fleet** and everyone else's latency collapses — the noisy-neighbour problem.

Three layers, cheapest first:

| Layer | Mechanism | Catches |
|---|---|---|
| **Request rate** | token bucket per API key | runaway loops, retry storms |
| **Token rate** | tokens/minute per tenant (input + output) | a few enormous requests |
| **Concurrency** | max in-flight requests per tenant | batch jobs monopolizing slots |

Rate-limiting *requests* alone is insufficient for LLMs, because one request can be 200k tokens. The
meaningful unit is **tokens**, and the meaningful cap is **concurrency**.

In [ ]:
def fairness(tenants, total_slots=64, policy="none"):
    # tenants: list of (name, offered_concurrency)
    offered = sum(t[1] for t in tenants)
    if policy == "none":
        alloc = [(n, total_slots * c / offered) for n, c in tenants]
    elif policy == "equal share":
        per = total_slots / len(tenants)
        alloc = [(n, min(c, per)) for n, c in tenants]
        # redistribute what the small tenants didn't use
        spare = total_slots - sum(a for _, a in alloc)
        hungry = [i for i, (n, c) in enumerate(tenants) if c > per]
        for i in hungry:
            alloc[i] = (alloc[i][0], alloc[i][1] + spare / len(hungry))
    elif policy == "weighted (by plan)":
        weights = {"enterprise": 4.0, "pro": 2.0, "free": 0.5}
        w = [weights.get(n.split()[-1], 1.0) for n, _ in tenants]
        tw = sum(w)
        alloc = [(tenants[i][0], min(tenants[i][1], total_slots * w[i] / tw))
                 for i in range(len(tenants))]
    return alloc

TENANTS = [("acme enterprise", 8), ("globex pro", 6), ("initech pro", 4),
           ("batchco free", 120)]     # <- the noisy neighbour
print("64 slots. 'batchco' fires a 120-way batch job while three paying tenants work:\n")
for policy in ("none", "equal share", "weighted (by plan)"):
    alloc = fairness(TENANTS, policy=policy)
    print(f"{policy:<22}" + "  ".join(f"{n.split()[0]}:{a:>5.1f}" for n, a in alloc))
print("\nWith no policy, batchco takes ~55 of 64 slots and every paying customer's")
print("latency degrades - the classic 'why did our p99 triple on Tuesday' incident.")
print("\nImplementation note: enforce this in your GATEWAY, not the engine. vLLM schedules")
print("fairly among admitted requests but has no concept of tenants (nb 30) - it cannot")
print("know that 120 of the requests in its queue belong to one customer.")

## Part 4 · Graceful drain and rollout

Deploying a new model or config means taking replicas down — while requests are **mid-stream**.

**The drain sequence** (each step matters):

```
1. mark the replica unhealthy in the LB     → no NEW requests routed here
2. keep serving in-flight requests          → existing streams complete normally
3. wait for num_requests_running → 0        → or a hard deadline (say 2× your p99)
4. SIGTERM the process                      → clean shutdown
5. start the new version                    → and WAIT for it to be ready
6. mark healthy only after a real request succeeds
```

**Step 6 is the one people skip.** A vLLM server's port opens well before the engine is ready — the
startup log (nb 26) shows model loading, KV profiling, and CUDA graph capture taking **tens of
seconds to minutes**. A health check that only tests TCP connectivity will route traffic into a
replica that isn't serving yet.

| Check | Tests | Verdict |
|---|---|---|
| TCP connect | the port is open | ❌ passes far too early |
| `GET /health` | the server process is up | 🟡 better; still may precede full readiness |
| `GET /v1/models` | the engine registered the model | ✅ good readiness signal |
| A real 1-token completion | the whole path works | ✅ best; use for the first probe after start |

**Capacity during rollout** is the part that bites: rolling one replica at a time out of N means you
run at `(N−1)/N` capacity for the whole deploy. If you were already at 80% utilization with 4
replicas, a rolling deploy puts you at 107% — an outage you scheduled yourself.

In [ ]:
def rollout_capacity(n_replicas, utilization, surge=0):
    during = (n_replicas - 1 + surge) / n_replicas
    load_during = utilization / during
    return {"replicas": n_replicas, "capacity_during": during,
            "load_during": load_during, "safe": load_during < 1.0}

print("Rolling deploy, one replica at a time:\n")
print(f"{'replicas':>9}{'normal load':>13}{'capacity during':>17}{'load during':>13}   verdict")
print("-" * 72)
for n in (2, 3, 4, 8, 16):
    for util in (0.5, 0.7, 0.8):
        r = rollout_capacity(n, util)
        if util == 0.7 or (n <= 4 and util == 0.8):
            print(f"{n:>9}{util:>12.0%}{r['capacity_during']:>17.0%}{r['load_during']:>12.0%}"
                  f"   {'✅ fine' if r['safe'] else '❌ OVERLOAD during deploy'}")

print("\nFixes, in order of preference:")
print("  1. SURGE first: start the new replica BEFORE draining the old one (needs spare quota)")
print("  2. deploy during a traffic trough (know your daily curve)")
print("  3. keep steady-state utilization under (N-1)/N - this is the same N-1 rule as nb 29")
print("\nAnd always: canary ONE replica, watch nb 26's metrics for a few minutes,")
print("then continue. The metrics that catch a bad model config fastest are KV usage")
print("(a config change can silently shrink your pool) and TTFT p95.")

## Part 5 · The failure taxonomy

What actually breaks in LLM serving, what it looks like, and what to do:

| Failure | Signature | Immediate action | Prevention |
|---|---|---|---|
| **OOM at startup** | dies right after `Available KV cache memory:` | lower `--gpu-memory-utilization` | capacity math from the startup log (nb 26) |
| **OOM at runtime** | crash under a long-prompt burst | cap `--max-model-len`, `--max-num-seqs` | leave headroom; the pool is not the limit, the peak is |
| **KV exhaustion** | KV ~99%, preemptions, throughput collapse | shed load, restart if thrashing | nb 26's alerts on KV + queue |
| **Silent slow path** | throughput 3× worse after a deploy | roll back, then bisect the config | benchmark every config change (nb 27) |
| **GPU fallen off the bus** | `nvidia-smi`/`rocm-smi` errors, ECC failures | drain and replace the node | node health checks; auto-cordon |
| **NaN / inf in logits** | garbage or empty outputs | roll back the model or dtype | validate after every quantization change (nb 23) |
| **Tokenizer mismatch** | subtly wrong output, wrong token counts | roll back | pin tokenizer with the checkpoint |
| **Cancellation leak** | queueing that doesn't match traffic | fix proxy buffering | Part 1's disconnect test |
| **Retry storm** | load spike right after an incident | 429 + `Retry-After`, jittered backoff | bounded queue (Part 2) |
| **Noisy neighbour** | one tenant's p99 damages everyone | per-tenant concurrency cap | Part 3 |

**The generic rule:** an LLM server can be *unhealthy but responsive*. Design health checks that
test the **serving path**, not the process.

## Part 6 · A chaos drill

The most valuable reliability exercise: **kill a replica at peak load and see what happens**. Let's
simulate it — with and without the N−1 headroom that notebook 29 recommended.

In [ ]:
def chaos_drill(n_replicas, per_replica_capacity, arrival_rate, fail_at=60,
                recovery_at=180, sim_seconds=300, slo_wait=5.0):
    queue, t = [], 0
    dt, rows = 1.0, []
    breached = 0
    while t < sim_seconds:
        alive = n_replicas - (1 if fail_at <= t < recovery_at else 0)
        capacity = alive * per_replica_capacity
        n_arrive = int(arrival_rate)
        queue.extend([t] * n_arrive)
        served = 0
        while queue and served < capacity:
            born = queue.pop(0); served += 1
        wait = (t - queue[0]) if queue else 0.0
        if wait > slo_wait: breached += 1
        rows.append({"t": t, "alive": alive, "queue": len(queue), "wait": round(wait, 1),
                     "capacity": capacity})
        t += dt
    return rows, breached

PER_REP, N_REPLICAS = 6.0, 4
print(f"A 4-replica fleet, {PER_REP} req/s each = {N_REPLICAS*PER_REP:.0f} req/s total capacity.")
print(f"One replica dies at t=60s and returns at t=180s, leaving {(N_REPLICAS-1)*PER_REP:.0f} req/s.\n")
print(f"{'offered load':>13}{'normal util':>13}{'util during':>13}{'peak queue':>12}"
      f"{'peak wait':>11}{'SLO breach':>12}")
print("-" * 76)
drills = {}
for arrival in (12.0, 15.0, 17.0, 18.0, 19.0, 21.0):
    rows, breached = chaos_drill(N_REPLICAS, PER_REP, arrival)
    drills[arrival] = rows
    peak_q = max(r["queue"] for r in rows)
    peak_w = max(r["wait"] for r in rows)
    util = arrival / (N_REPLICAS * PER_REP)
    util_down = arrival / ((N_REPLICAS - 1) * PER_REP)
    print(f"{arrival:>12.0f}/s{util:>12.0%}{util_down:>12.0%}{peak_q:>12}"
          f"{peak_w:>10.0f}s{breached:>10}s")

print("\nThe threshold is sharp and it is arithmetic, not luck: survival depends entirely on")
print(f"whether N-1 capacity ({(N_REPLICAS-1)*PER_REP:.0f} req/s) still exceeds your offered load.")
print("Below it, the queue drains and users barely notice. Above it, the queue GROWS for the")
print("whole outage and keeps growing until capacity returns - the backlog is a debt you")
print("repay at (capacity - arrival) per second, long after the incident 'ended' (nb 26).")
print("\nThat is what 'provision for N-1' buys, and why notebook 27 said to run at ~70% of")
print("measured peak: at 75% normal utilization here, losing one of four still just fits.")

In [ ]:
viz = {"series": [{"n": n, "rows": drills[n]} for n in sorted(drills)],
       "fail_at": 60, "recovery_at": 180}

JS = r'''
const M = {top: 20, right: 90, bottom: 40, left: 56};
const iw = W - M.left - M.right, ih = H - M.top - M.bottom;
const svg = root.append("svg").attr("width",W).attr("height",H).append("g")
    .attr("transform",`translate(${M.left},${M.top})`);
const all = data.series.flatMap(s=>s.rows);
const x = d3.scaleLinear().domain(d3.extent(all,d=>d.t)).range([0,iw]);
const y = d3.scaleLinear().domain([0, d3.max(all,d=>d.queue)*1.1 || 1]).range([ih,0]);
svg.append("g").attr("transform",`translate(0,${ih})`).call(d3.axisBottom(x).ticks(8));
svg.append("g").call(d3.axisLeft(y).ticks(5));
svg.append("text").attr("x",iw/2).attr("y",ih+34).attr("text-anchor","middle")
   .style("font-size","12px").text("time (s)");
svg.append("text").attr("transform","rotate(-90)").attr("x",-ih/2).attr("y",-42)
   .attr("text-anchor","middle").style("font-size","12px").text("requests queued");

svg.append("rect").attr("x",x(data.fail_at)).attr("width",x(data.recovery_at)-x(data.fail_at))
   .attr("y",0).attr("height",ih).attr("fill","#ffcdd2").attr("opacity",0.45);
svg.append("text").attr("x",x(data.fail_at)+6).attr("y",14).style("font-size","11px")
   .style("fill","#c62828").text("one replica down");

const color = d3.scaleOrdinal().domain(data.series.map(s=>s.n))
    .range(["#c62828","#ef6c00","#2e7d32","#1565c0"]);
const line = d3.line().x(d=>x(d.t)).y(d=>y(d.queue));
svg.selectAll("s").data(data.series).join("path")
   .attr("fill","none").attr("stroke",d=>color(d.n)).attr("stroke-width",2)
   .attr("d",d=>line(d.rows))
   .append("title").text(d=>`${d.n} replicas`);
const labs = data.series.map(s=>({n:s.n, ly:y(s.rows[s.rows.length-1].queue)}))
                        .sort((a,b)=>a.ly-b.ly);
for (let i=1;i<labs.length;i++) if (labs[i].ly-labs[i-1].ly<13) labs[i].ly=labs[i-1].ly+13;
svg.selectAll("l").data(labs).join("text")
   .attr("x",iw+8).attr("y",d=>d.ly+4).style("font-size","11px")
   .style("fill",d=>color(d.n)).text(d=>`${d.n} replicas`);
'''
show_d3(JS, viz, height=330)

## Part 7 · The runbook

**Before you go live**

- [ ] Cancellation verified with the disconnect test (Part 1)
- [ ] Queue bounded; 429 + `Retry-After` on rejection (Part 2)
- [ ] Per-tenant concurrency and token limits in the gateway (Part 3)
- [ ] Readiness probe hits `/v1/models` or a real completion — never just TCP (Part 4)
- [ ] Drain sequence tested: no dropped streams during a deploy
- [ ] Steady-state utilization below `(N−1)/N` (Parts 4 & 6)
- [ ] Alerts on **KV usage** and **queue depth**, paging on TTFT (nb 26)
- [ ] A chaos drill actually run, in staging at minimum

**When it's on fire**

| Symptom | First move |
|---|---|
| TTFT climbing, KV high | shed load (lower queue limit); check for a long-prompt burst |
| Throughput collapsed, preemptions | restart the replica; it may be thrashing (nb 26) |
| One tenant's traffic spiked | apply their concurrency cap; don't scale the fleet for one caller |
| After a deploy | **roll back first, diagnose second** |
| Queueing that doesn't match traffic | suspect the cancellation leak (Part 1) |
| One replica slower than its peers | cordon it; compare per-replica metrics, check GPU health |

**Post-incident**: every incident should produce either an alert that would have caught it earlier
(a *cause* metric, not a symptom) or a limit that would have contained it.

## Recap

1. **Unhandled cancellation silently eats capacity** and looks like healthy throughput.
2. **Bounded queues + honest 429s beat unbounded queues** — the unbounded queue serves *fewer*
   users successfully while wasting GPU on already-timed-out work.
3. **Rate-limit tokens and concurrency, not just requests** — one request can be 200k tokens, and
   the engine has no idea what a tenant is.
4. **Readiness ≠ liveness.** A vLLM port opens minutes before the engine can serve.
5. **Rolling deploys cost you a replica.** If you can't survive N−1, you can't deploy safely.
6. **Run the chaos drill.** The difference between 3 and 5 replicas isn't cost — it's whether a
   single failure is a page or a shrug.

### Further reading
- [Google SRE Book](https://sre.google/sre-book/table-of-contents/) — especially *Handling Overload* and *Addressing Cascading Failures*
- [Stop Rate Limiting! Capacity Management Done Right](https://www.youtube.com/watch?v=m64SWl9bfvk) (Rodrigo Schaefer / Jon Moore) — why queue limits beat rate limits
- Prerequisites: nb [26](./Serving_Logs_Observability.ipynb) (the metrics), [27](./Serving_Benchmark_Capacity_Planning.ipynb) (the knee and headroom), [29](./Distributed_MultiReplica_Serving.ipynb) (N−1 planning), [30](./MultiLoRA_Serving_At_Scale.ipynb) (tenancy)